# Cảnh báo KPI theo ngày: ngưỡng có mùa vụ thay cho ngưỡng cố định

Bộ `bike-sharing-uci`, tệp `day.csv`: 731 ngày thuê xe Capital Bikeshare (Washington DC) từ 01/01/2011 tới
31/12/2012, cột `cnt` là tổng lượt thuê mỗi ngày, kèm `holiday` và `weathersit` (1 trời đẹp ... 3 mưa/tuyết, 4 rất xấu).

**Bối cảnh.** Head of Operations muốn mỗi sáng nhận một tin nhắn khi lượt thuê hôm qua "bất thường", để kịp hỏi
đội điều phối. Bản đầu tiên team dùng ngưỡng cố định ("dưới 2 000 lượt thì báo") và bị bỏ qua sau hai tuần: mùa
đông báo liên tục, mùa hè thì một ngày mất nửa lượt thuê cũng không báo. Bạn làm bản thứ hai:

- **baseline có mùa vụ**: median của cùng thứ trong 4 tuần trước;
- **điểm bất thường robust**: độ lệch log so với baseline, chia cho MAD của chính độ lệch đó trong 8 tuần trước;
- **gộp cảnh báo** thành đợt và **phân loại** (lễ, thời tiết xấu, cần điều tra) để tin nhắn đọc được trong 10 giây.

**Cách làm bài**
- Cell chấm điểm chỉ định nghĩa hàm. Điền vào chỗ `# TODO`; hàm không được sửa dữ liệu đầu vào.
- Cell gắn tag `explore` (dòng đầu `# explore: ...`) đọc dữ liệu thật và vẽ; khi chấm chúng bị bỏ qua.
- Chấm: `uv run pytest tracks/data-analyst/modules/13-business-financial-analytics/exercises/basic/03-kpi-anomaly-alerts -q`

In [ ]:
# explore: tìm gốc repo, tải bộ dữ liệu nếu chưa có, đọc day.csv
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT))
from data.download_dataset import ensure
import pandas as pd

day = pd.read_csv(ensure("bike-sharing-uci") / "day.csv", parse_dates=["dteday"]).set_index("dteday")
day.index.name = "date"
y = day["cnt"].astype(float).rename("cnt")
print(len(y), y.index.min().date(), "→", y.index.max().date(), "| đủ mọi ngày:", y.index.equals(pd.date_range(y.index.min(), y.index.max())))
print(y.describe().round(0))

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

## 1. Baseline có mùa vụ

`seasonal_baseline(y, weeks=4) -> pd.Series`: `y` là Series theo ngày, index liên tục (không thiếu ngày). Giá trị
ngày t là **median** của `y` ở t − 7, t − 14, ..., t − 7·`weeks` (cùng thứ trong `weeks` tuần trước). Ngày chưa đủ
`weeks` tuần lịch sử thì `NaN`. Cùng index với `y`, tên Series `baseline`.

Vì sao median cùng thứ: lượt thuê thứ Bảy khác thứ Hai, và một ngày bão trong 4 tuần trước không được kéo
baseline xuống (mean thì bị kéo).

In [ ]:
def seasonal_baseline(y: pd.Series, weeks: int = 4) -> pd.Series:
    """Median của cùng thứ trong `weeks` tuần trước; NaN khi chưa đủ lịch sử; tên "baseline"."""
    # TODO: y.shift(7 * k) với k = 1..weeks, ghép theo cột, median theo dòng (chưa đủ tuần thì NaN)
    raise NotImplementedError

## 2. Điểm bất thường robust

`robust_score(y, baseline, window=56, min_periods=28) -> pd.Series` (tên `score`):

1. độ lệch log `r = ln(y / baseline)` (giảm một nửa và tăng gấp đôi có cùng độ lớn);
2. thang đo của ngày t: `1.4826 × MAD` của `r` trong `window` ngày **trước** t (không gồm t), bỏ qua `NaN`, cần ít
   nhất `min_periods` giá trị không `NaN`, nếu không thì `NaN`. MAD là median của `|r − median(r)|`; nhân 1.4826 để
   gần với độ lệch chuẩn khi dữ liệu chuẩn;
3. `score = r / thang đo`.

Không dùng chính ngày t để tính thang đo: một ngày bão làm thang đo của chính nó phình ra và tự che mình.

In [ ]:
def robust_score(y: pd.Series, baseline: pd.Series, window: int = 56, min_periods: int = 28) -> pd.Series:
    """ln(y / baseline) chia cho 1.4826 * MAD của nó trong `window` ngày trước; tên "score"."""
    # TODO: ratio = log(y / baseline); thang đo = 1.4826 * MAD của ratio.shift(1) trên cửa sổ trượt (np.nanmedian)
    raise NotImplementedError

## 3. Bảng cảnh báo, gộp đợt, phân loại

- `alert_table(y, baseline, score, threshold=4.0) -> pd.DataFrame`: các ngày `|score| > threshold` (score `NaN` thì
  không báo), index `date` tăng dần, cột `value`, `baseline`, `score`, `direction` (`"drop"` khi score âm, `"spike"`
  khi dương).
- `alert_episodes(alerts) -> pd.DataFrame`: gộp các ngày cảnh báo **liên tiếp** (cách nhau đúng 1 ngày) thành một
  đợt. Cột `start`, `end`, `days`, `worst_score` (score có trị tuyệt đối lớn nhất trong đợt, giữ dấu); index
  `0, 1, ...` theo thời gian. Không có cảnh báo thì trả bảng rỗng có đủ bốn cột.
- `triage(alerts, day) -> pd.DataFrame`: bản sao `alerts` thêm cột `reason`: `"holiday"` nếu `day.holiday == 1`, nếu
  không thì `"bad_weather"` nếu `day.weathersit >= 3`, còn lại `"investigate"`. `day` có index ngày như `alerts`.

In [ ]:
def alert_table(y: pd.Series, baseline: pd.Series, score: pd.Series, threshold: float = 4.0) -> pd.DataFrame:
    """Các ngày |score| > threshold: cột value, baseline, score, direction; index "date"."""
    # TODO
    raise NotImplementedError


def alert_episodes(alerts: pd.DataFrame) -> pd.DataFrame:
    """Gộp ngày liên tiếp thành đợt: start, end, days, worst_score."""
    # TODO: đánh số đợt bằng (ngày - ngày trước) != 1 ngày, rồi cumsum
    raise NotImplementedError


def triage(alerts: pd.DataFrame, day: pd.DataFrame) -> pd.DataFrame:
    """Thêm cột reason: holiday > bad_weather (weathersit >= 3) > investigate."""
    # TODO
    raise NotImplementedError

In [ ]:
# explore: hai quy tắc ngây thơ - ngưỡng cố định và giảm hơn 30% so với hôm qua
fixed = y[y < 2000]
dod = y[y.pct_change() < -0.30]
print(f"ngưỡng cố định < 2 000: {len(fixed)} ngày; theo tháng:", fixed.index.month.value_counts().sort_index().to_dict())
print(f"giảm > 30% so với hôm qua: {len(dod)} ngày")

In [ ]:
# explore: quy tắc mùa vụ trên dữ liệu thật
base = seasonal_baseline(y)
score = robust_score(y, base)
alerts = triage(alert_table(y, base, score), day)
episodes = alert_episodes(alerts)
print(f"ngày có score: {score.notna().sum()} (từ {score.first_valid_index().date()})")
print(f"cảnh báo |score| > 4: {len(alerts)} ngày ({(alerts.direction == 'drop').sum()} drop, "
      f"{(alerts.direction == 'spike').sum()} spike), {len(episodes)} đợt")
print(alerts.reason.value_counts().to_dict())
print(alerts.reindex(alerts.score.abs().sort_values(ascending=False).index).head(10).round(1))
for k in [3, 3.5, 4, 5]:
    print(f"ngưỡng {k}: {int((score.abs() > k).sum())} ngày")

In [ ]:
# explore: chart cho tin nhắn buổi sáng - giá trị, baseline, ngày cảnh báo
fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(y.index, y, color="#9aa5b1", lw=0.8, label="lượt thuê")
ax.plot(base.index, base, color="#4c78a8", lw=1.2, label="baseline (median cùng thứ 4 tuần)")
drops = alerts[alerts.direction == "drop"]
ax.scatter(drops.index, drops.value, color="#e45756", s=18, zorder=3, label="cảnh báo drop")
ax.set_ylim(0, None)
ax.spines[["top", "right"]].set_visible(False)
ax.legend(loc="upper left", frameon=False)
ax.set_title("Cảnh báo theo baseline mùa vụ: tập trung vào ngày lệch khỏi cùng thứ các tuần trước", loc="left")
plt.show()

In [ ]:
# explore: các đợt cần điều tra (không phải lễ, không phải thời tiết xấu)
todo = alerts[alerts.reason == "investigate"]
print(todo.join(day[["weathersit", "temp", "hum", "windspeed"]]).round(2))

## 4. Đọc kết quả, rồi làm trắc nghiệm

1. Ngưỡng cố định báo bao nhiêu ngày, rơi vào mùa nào? Vì sao Head of Operations bỏ qua nó?
2. Quy tắc mùa vụ báo bao nhiêu ngày, gộp thành bao nhiêu đợt? Bao nhiêu phần có lý do "hiển nhiên" (lễ, thời
   tiết xấu)? Đợt lớn nhất là gì (tra lịch sử Washington DC cuối tháng 10/2012)?
3. Thử ngưỡng 3 và 5: số cảnh báo đổi thế nào? Bạn chọn ngưỡng nào cho tin nhắn hằng ngày, và vì sao?
4. Viết mẫu tin nhắn một dòng cho một cảnh báo (giá trị, baseline, % lệch, lý do, ai cần làm gì).

Xong thì mở app, làm phần **trắc nghiệm** của bài.